In [ ]:
import pandas as pd
import numpy as np
!pip install openpyxl
import seaborn as sns
import matplotlib.pyplot as plt

# Cleaning Data

In [ ]:
xl = pd.ExcelFile(r"INSERT_FILE_PATH")
print(xl.sheet_names)
df_baseline = pd.read_excel(r"INSERT_FILE_PATH", sheet_name='Benchmarking', header=0)
pd.set_option('display.max_rows', None)
df_baseline= df_baseline.drop(columns = [151, 'Unnamed: 18', 'Total # of Progress Notes ','Notes '])

#Only take KN's data for analysis:
df_baseline['Entry #'] = df_baseline['Entry #'].ffill()
df_baseline = df_baseline[df_baseline["Entry #"] >=10]
#Remove Example Entry
df_baseline = df_baseline[df_baseline['Entry #'] != 100].reset_index(drop=True)
df_baseline = df_baseline.iloc[:143]
df_baseline.columns = df_baseline.columns.str.strip()

In [ ]:
#Strip White Space:
df_baseline['RN or LPN?'] = df_baseline['RN or LPN?'].str.strip()
df_baseline['FT, PT, or C?'] = df_baseline['FT, PT, or C?'].str.strip()

#Forward Fills Identifier Columns
col_to_fill = ['Date', 'Nurse ID #', 'Time Block']
df_baseline[col_to_fill] = df_baseline[col_to_fill].ffill()
#Forward fills Nurses Types + Shift only when the total charting time also occurs
condition = df_baseline["Total Time per Progress Note"].notnull()
nurse_shift_fill = ['RN or LPN?', 'FT, PT, or C?']
df_baseline[nurse_shift_fill] = df_baseline[nurse_shift_fill].ffill().where(condition)

# Covert Date Time columns to String Format for SPSS
df_baseline['Date'] = pd.to_datetime(df_baseline['Date'], dayfirst=True).dt.strftime('%d-%m-%Y')

In [ ]:
# Export to new Excel Files
df_baseline.to_csv("KN_Baseline_Observational_Data_SPSS_Analysis", index = False) #

# Remove Outliers in Total Charting Time per Notes column


In [ ]:
df_baseline_copy = df_baseline.copy() #Create copy of original DF for outliers removal
df_baseline_copy.dtypes
df_baseline_copy.columns

In [ ]:
# Function to convert everything date time to integer:
def convert_int_time(col):
    col_seconds = pd.to_timedelta(col.astype(str)).dt.total_seconds()
    col_minute = col_seconds / 60
    return col_minute

col_to_apply = ['Total Time per Progress Note']
df_baseline_copy[col_to_apply] = df_baseline_copy[col_to_apply].apply(convert_int_time)

#Plot to visualize the outliers before removal
plt.style.use('default')
fig, ax = plt.subplots(facecolor='white')
sns.boxplot(df_baseline_copy['Total Time per Progress Note'],
            color = 'steelblue',
            medianprops = dict(color='orange', linewidth=2),
            whiskerprops=dict(color='black', linewidth=1.5),
            capprops=dict(color='black', linewidth=1.5),
            boxprops=dict(edgecolor='black'),
            flierprops=dict(markerfacecolor='gray', marker='o'),
            ax=ax)
ax.set_facecolor('white')
ax.set_title('Boxplot of Total Time per Progress Note')
plt.show()

In [ ]:
#Fill null values as 0
df_baseline_copy = df_baseline_copy.fillna(0)
# Remove 25 minutes mark and visualize distribution:
def remove_outliers(df, col):
    threshold = 10 #Original outliers were 12 minutes and 25 minutes, we want to remove this (Won't remove 8 minutes because it falls within typical range of nurse's charting time -> Context matters)
    removed_outliers = df[df[col] <= threshold]
    return removed_outliers

df_baseline_copy_no_outliers = remove_outliers(df_baseline_copy, 'Total Time per Progress Note')
df_baseline_copy_no_outliers['Total Time per Progress Note'].unique()

#Plot to visualize the outliers before removal
plt.style.use('default')
fig, ax = plt.subplots(facecolor='white')
sns.boxplot(df_baseline_copy_no_outliers['Total Time per Progress Note'],
            color = 'steelblue',
            medianprops = dict(color='orange', linewidth=2),
            whiskerprops=dict(color='black', linewidth=1.5),
            capprops=dict(color='black', linewidth=1.5),
            boxprops=dict(edgecolor='black'),
            flierprops=dict(markerfacecolor='gray', marker='o'),
            ax=ax)
ax.set_facecolor('white')
ax.set_title('Boxplot of Total Time per Progress Note')
plt.show()

#Export DataFrame for Analysis without outliers
df_baseline_copy_no_outliers.to_csv("baseline_clean_data_ffill_NurseShift_Types_Outliers_Removed", index = False) # Export for analysis without outliers